# 00 · Data preparation with Google Earth Engine (Python)

Same steps as the browser scripts in `gee_scripts/`, but run from VS Code:

1. Flood mapping with Sentinel-1 for the 3 events → flood maps + flood / non-flood points
2. 14 conditioning factors at every point → **one CSV dataset**
3. Factor stacks for Hyderabad / Warangal → used by `03_maps.ipynb` to draw maps **in Python**

Results are exported to **Google Drive → `flood_project`**; download them into `data/` afterwards.

> You only need this notebook if you want to re-create the data. Notebooks 01–03 work from the downloaded files.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import ee
import geemap
from src.config import EVENTS, APPLICATION_AREAS, GEE_COLUMNS
from src import gee_lib as G

G.init()   # first time: a browser window opens to log in to Google

## 1 · Flood mapping (one event at a time)
`EVENT = 0` Godavari 2022 · `1` Munneru 2024 · `2` Budameru 2024

In [ ]:
EVENT = 0
ev = EVENTS[EVENT]
res = G.flood_inventory(ev)

info = ee.Dictionary(res['info']).getInfo()
for k, v in info.items():
    print(f'{k:20s}: {v}')

In [ ]:
# Interactive map (pre-flood radar, flood radar, detected flood, points)
m = geemap.Map(basemap='HYBRID')
m.centerObject(res['aoi'], 10)
m.addLayer(res['pre'], {'min': -25, 'max': 0}, 'Pre-flood VV (dB)', False)
m.addLayer(res['post'], {'min': -25, 'max': 0}, 'Flood VV (dB)', False)
m.addLayer(res['permanent_water'].selfMask().clip(res['aoi']), {'palette': ['000080']}, 'Permanent water', False)
m.addLayer(res['flood'].selfMask(), {'palette': ['00BFFF']}, 'FLOOD')
m

In [ ]:
# Export the flood map and the points to Google Drive (Tasks run in the background)
G.export_image(res['flood'].unmask(255).toByte(), f"{ev['name']}_flood_map", res['aoi'])
G.export_table(res['points'], f"{ev['name']}_flood_points", file_format='SHP')

## 2 · Training dataset (14 factors at every point)
Uses the point assets you already uploaded to GEE. Change the paths if your asset names differ.

In [ ]:
POINT_ASSETS = {
    'godavari_2022': 'projects/flood-mapping-511811/assets/godavari_2022_flood_points',
    'munneru_2024':  'projects/flood-mapping-511811/assets/munneru_2024_flood_points',
    'budameru_2024': 'projects/flood-mapping-511811/assets/budameru_2024_flood_points',
}

samples = ee.FeatureCollection([])
for ev in EVENTS:
    s = G.sample_factors(ee.FeatureCollection(POINT_ASSETS[ev['name']]), ev['flood_date'])
    print(ev['name'], '→ samples:', s.size().getInfo(),
          ' per class:', s.aggregate_histogram('label').getInfo())
    samples = samples.merge(s)

print('TOTAL:', samples.size().getInfo())
# Check drainage density is no longer all zeros
print('drain_dens min / max:',
      samples.aggregate_min('drain_dens').getInfo(), '/', samples.aggregate_max('drain_dens').getInfo())

In [ ]:
G.export_table(samples, 'flood_dataset_all_events',
               selectors=['event', 'lon', 'lat'] + GEE_COLUMNS + ['label'])

## 3 · Factor stacks for Python mapping
Exports one 14-band GeoTIFF per area (rainfall = scenario from `src/config.py`).
Download them into `data/stacks/`. Hyderabad ≈ 150 MB.

In [ ]:
for name, area in APPLICATION_AREAS.items():
    region = G.rect(area['aoi'])
    stack = G.factor_stack(region).toFloat().clip(region)
    G.export_image(stack, f'{name}_factor_stack', region, scale=area['scale'])

# Optional: stacks for the 3 study areas with their REAL event rainfall
# for ev in EVENTS:
#     region = G.rect(ev['aoi'])
#     stack = G.factor_stack(region, ev['flood_date']).toFloat().clip(region)
#     G.export_image(stack, f"{ev['name']}_factor_stack", region)

In [ ]:
# Check progress of all export tasks
for t in ee.batch.Task.list()[:10]:
    print(t.status()['description'], '→', t.status()['state'])